In [76]:
import sqlite3                                 
import pandas as pd                             
 
con = sqlite3.connect('../data/raw/airtel.db')  
 
def q(sql):                                     
    return pd.read_sql(sql, con)

In [77]:
# A1 · How many subscribers are prepaid and 
# how many postpaid, and what share is each?
# STEPS
# 1. Open customer_features, where one row is one subscriber.
# 2. Group by connection_type.
# 3. Count the rows in each group.
# 4. Express each count as a share of 22,000.
q("SELECT connection_type, COUNT(*) AS count, COUNT(*) * 100.0 /"
" (SELECT COUNT(*) FROM customer_features)" \
" AS share FROM customer_features GROUP BY connection_type")

,connection_type,count,share
0,Postpaid,4851,22.05
1,Prepaid,17149,77.95


In [78]:
#A2 · Which five plans have the most subscribers, and what does each cost?
#STEPS
#1. Group customer_features by plan_name.
#2. Carry plan_type and price_inr through the group.
#3. Count and sort, biggest first.
#4. Show the top five.
q("SELECT plan_name, plan_type, price_inr, COUNT(*) AS count FROM customer_features")


,plan_name,plan_type,price_inr,count
0,Prepaid Data 349,Prepaid,349,22000


In [79]:
#A2 · Which five plans have the most subscribers, and what does each cost?
#STEPS
#1. Group customer_features by plan_name.
#2. Carry plan_type and price_inr through the group.
#3. Count and sort, biggest first.
#4. Show the top five.
q("""SELECT plan_name, plan_type, price_inr, COUNT(*) AS subs
FROM customer_features
GROUP BY plan_name, plan_type, price_inr
ORDER BY subs DESC
LIMIT 5""")

,plan_name,plan_type,price_inr,subs
0,Prepaid Annual 1799,Prepaid,1799,2901
1,Prepaid Quarterly 719,Prepaid,719,2561
2,Prepaid Max 479,Prepaid,479,2392
3,Prepaid Entertainment 399,Prepaid,399,2142
4,Prepaid Data 349,Prepaid,349,1884


In [80]:
#A3 · Which five circles have the most subscribers?
#STEPS
#1. Group by circle_name, carrying region.
#2. Count.
#3. Sort and take five.
q("""SELECT circle_name, region, COUNT(*) AS subs
FROM customer_features
GROUP BY circle_name, region
ORDER BY subs DESC
LIMIT 5""")

,circle_name,region,subs
0,Maharashtra,West,1428
1,Andhra Pradesh,South,1412
2,Mumbai,West,1395
3,Karnataka,South,1361
4,North East,East,1318


In [81]:
#A4 · How many subscribers are in each age band?
#1. Group by age_band.
#2. Count and express as a share.
#3. Sort by band, not by size, so the shape of the distribution is visible.
q(""" SELECT age_band,COUNT(*) AS subs,
ROUND(100.0*COUNT(*)/22000)AS perc
FROM Customer_features
GROUP BY age_band
ORDER BY subs DESC""")

,age_band,subs,perc
0,25-34,7467,34.0
1,18-24,5268,24.0
2,35-44,4824,22.0
3,45-54,2895,13.0
4,55+,1546,7.0


In [82]:
#A5 · How many subscribers did each acquisition channel bring in?
#1. Group by acquisition_channel.
#2. Count and share.
#3. Sort biggest first.
q("""SELECT acquisition_channel,COUNT(*) AS subs,
ROUND (100.0*COUNT(*)/22000) AS perc
FROM customer_features
GROUP BY acquisition_channel
ORDER BY subs DESC""")


,acquisition_channel,subs,perc
0,Retailer,9287,42.0
1,Airtel App,6513,30.0
2,Website,3081,14.0
3,Call Center,1770,8.0
4,Doorstep,1349,6.0


In [83]:
# A6 · How many subscribers use each device type and operating system?
# STEPS
# 1. Group by device_type and os together.
# 2. Count.
# 3. Read the result and notice that only three of the
# six possible combinations exist.
q("""SELECT device_type, os, COUNT(*) AS subs
FROM customer_features
GROUP BY device_type, os
ORDER BY subs DESC""")


,device_type,os,subs
0,Smartphone,Android,15613
1,Smartphone,iOS,3358
2,Feature Phone,Other,3029


In [84]:
#A7 · How many subscribers have 0, 1, 2 and 3 completed years with us?
#1. Group by tenure_years, which Step 2 calculated from activation_date.
#2. Count and share.
#3. Sort by year.
q("""
SELECT 
    tenure_years,COUNT(*) AS subs,
    ROUND((COUNT(*) * 100.0 / 22000), 2) AS perc
FROM customer_features
GROUP BY tenure_years
ORDER BY subs DESC;""")

,tenure_years,subs,perc
0,2,7055,32.07
1,1,6961,31.64
2,0,4610,20.95
3,3,3374,15.34


In [85]:
#B1 · What is our average monthly revenue per subscriber,
 #by the company's ARPU figure? — 

#STEPS
#1. Open customer_features.
#2. Average stated_arpu.
#3. Report it.

q("SELECT ROUND(AVG(stated_arpu), 2) AS avg_arpu " \
"FROM customer_features")

,avg_arpu
0,421.65


In [86]:
# B2 · What is our average monthly revenue per subscriber, from the payments received?
# STEPS
# 1. Open customer_features again.
# 2. Average actual_arpu_6m, which Step 2 built from recharges.
# 3. Also total it, to get the monthly revenue of the whole base.
q("""
SELECT ROUND(AVG(actual_arpu_6m), 2) AS avg_arpu,
    ROUND(SUM(actual_arpu_6m), 2) AS total_monthly_revenue
FROM customer_features
""")

,avg_arpu,total_monthly_revenue
0,344.46,7578210.14


In [87]:
#B3  By how much does the companys ARPU figure overstate the money received?
#STEPS
#1. Average both columns in one query.
# 2. Average the gap.
# 3. Express the difference as a percentage of the actual total, not of the stated one.
q("""
SELECT
    ROUND(AVG(stated_arpu), 2) AS avg_stated_arpu,
    ROUND(AVG(actual_arpu_6m), 2) AS avg_actual_arpu,
    ROUND(AVG(stated_arpu - actual_arpu_6m), 2) AS avg_gap,
    ROUND(100.0 * AVG(stated_arpu - actual_arpu_6m)
          / AVG(actual_arpu_6m),2) AS gap_pct_of_actual
FROM customer_features
""")

,avg_stated_arpu,avg_actual_arpu,avg_gap,gap_pct_of_actual
0,421.65,344.46,77.18,22.41


In [88]:
#B4 · Is that gap the same for prepaid and for postpaid subscribers?
#STEPS
#1. Group by connection_type.
#2. Average both columns inside each group.
#3. Read the two rows against each other rather than down the column.
q("""
SELECT connection_type,
    ROUND(AVG(stated_arpu), 2) AS avg_stated_arpu,
    ROUND(AVG(actual_arpu_6m), 2) AS avg_actual_arpu,
    ROUND(AVG(stated_arpu - actual_arpu_6m), 2) AS avg_gap
FROM customer_features
GROUP BY connection_type
""")

,connection_type,avg_stated_arpu,avg_actual_arpu,avg_gap
0,Postpaid,959.65,970.42,-10.77
1,Prepaid,269.46,167.40,102.06


In [89]:
#B4 · Is that gap the same for prepaid and for postpaid subscribers?
#STEPS
#1. Group by connection_type.
#2. Average both columns inside each group.
#3. Read the two rows against each other rather than down the column.
q("""
SELECT
    connection_type,
    COUNT(*) AS subscribers,
    ROUND(AVG(stated_arpu), 2) AS avg_stated_arpu,
    ROUND(AVG(actual_arpu_6m), 2) AS avg_actual_arpu,
    ROUND(AVG(stated_arpu - actual_arpu_6m), 2) AS avg_gap
FROM customer_features
GROUP BY connection_type
""")

,connection_type,subscribers,avg_stated_arpu,avg_actual_arpu,avg_gap
0,Postpaid,4851,959.65,970.42,-10.77
1,Prepaid,17149,269.46,167.40,102.06


In [90]:
q("""
SELECT plan_name,
count(*) AS subscribers,
    ROUND(SUM(actual_arpu_6m), 2) AS monthly_revenue, 
    ROUND(AVG(actual_arpu_6m), 2) AS revenue_per_subscriber
FROM customer_features
GROUP BY plan_name
ORDER BY monthly_revenue DESC
LIMIT 5
""")

,plan_name,subscribers,monthly_revenue,revenue_per_subscriber
0,Postpaid Infinity 1499,1431,2171756.20,1517.65
1,Postpaid Family 999,1239,1249245.94,1008.27
2,Postpaid 749,946,718958.94,760.00
3,Prepaid Entertainment 399,2142,671849.50,313.66
4,Prepaid Data 349,1884,520066.00,276.04


In [91]:
# B6 · Which five circles bring in the most actual monthly revenue?
q("""
SELECT circle_name, SUM(actual_arpu_6m) AS monthly_revenue
FROM customer_features
GROUP BY circle_name
ORDER BY monthly_revenue DESC
LIMIT 5
""")


,circle_name,monthly_revenue
0,Mumbai,473288.16
1,Karnataka,468514.95
2,Andhra Pradesh,463369.80
3,North East,460998.29
4,Maharashtra,450032.30


In [92]:
# B7 · Payments received from January through June 2025
q("""
SELECT COUNT(*) AS payments,
    ROUND(SUM(amount_inr), 2) AS total_received
FROM recharges
WHERE recharge_date >= '2025-01-01'
  AND recharge_date < '2025-07-01'
""")

,payments,total_received
0,83693,45469268.0


In [93]:
#B8 · Do subscribers who have been with us longer pay more each month?
#STEPS
#1. Group by tenure_years.
#2. Average actual_arpu_6m.
#3. Sort by year
q("""
SELECT tenure_years,
COUNT(*) AS subscriber_count,
AVG(actual_arpu_6m) AS avg_monthly_payment
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years
""")

,tenure_years,subscriber_count,avg_monthly_payment
0,0,4610,329.104857
1,1,6961,342.716430
2,2,7055,350.126465
3,3,3374,357.215611


In [94]:
q("""
SELECT txn_type,
    COUNT(*) AS payments,
    COUNT(DISTINCT customer_id) AS subscribers
FROM recharges
GROUP BY txn_type
""")

,txn_type,payments,subscribers
0,Bill Payment,108419,4851
1,Recharge,299670,17149


In [95]:
q("select avg(amount_inr)as averge_payment from recharges")

,averge_payment
0,532.58659


In [96]:
q("""
SELECT txn_type,
    ROUND(AVG(amount_inr), 2) AS average_payment,
    MIN(amount_inr) AS minimum_payment,
    MAX(amount_inr) AS maximum_payment
FROM recharges
GROUP BY txn_type
""")

,txn_type,average_payment,minimum_payment,maximum_payment
0,Bill Payment,1082.42,399,1873
1,Recharge,333.66,99,1799


In [97]:
q("""
SELECT connection_type,
    ROUND(AVG(recharge_count), 2) AS avg_payments,
    ROUND(AVG(recharge_total_6m), 2) AS avg_total_paid
FROM customer_features
GROUP BY connection_type
""")

,connection_type,avg_payments,avg_total_paid
0,Postpaid,5.39,5822.51
1,Prepaid,3.36,1004.39


In [98]:
q("""
SELECT channel,
    COUNT(*) AS payments,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM recharges), 2) AS share_pct
FROM recharges
GROUP BY channel
ORDER BY payments DESC
""")

,channel,payments,share_pct
0,Website,111163,27.24
1,Airtel App,111083,27.22
2,USSD,75257,18.44
3,Retailer,74406,18.23
4,AutoPay,36180,8.87


In [99]:
q("""
SELECT
    txn_type,
    channel,
    COUNT(*) AS no_of_payments,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY txn_type),
        2
    ) AS share_pct
FROM recharges
GROUP BY txn_type, channel
ORDER BY txn_type, no_of_payments DESC
""")

,txn_type,channel,no_of_payments,share_pct
0,Bill Payment,Airtel App,36186,33.38
1,Bill Payment,AutoPay,36180,33.37
2,Bill Payment,Website,36053,33.25
3,Recharge,USSD,75257,25.11
4,Recharge,Website,75110,25.06
5,Recharge,Airtel App,74897,24.99
6,Recharge,Retailer,74406,24.83


In [100]:
q("""
SELECT recharge_count,
    COUNT(*) AS subscribers,
    ROUND(AVG(is_churned*100), 2) AS avg_churn
FROM customer_features
WHERE recharge_count <= 8
GROUP BY recharge_count
ORDER BY recharge_count
""")

,recharge_count,subscribers,avg_churn
0,0,3711,44.30
1,1,2762,40.04
2,2,2339,28.35
3,3,1026,54.09
4,4,714,76.05
5,5,3407,13.77
6,6,6732,1.52
7,7,507,5.52
8,8,32,90.63


In [101]:
q("""
SELECT month,
COUNT(*) AS subs,
ROUND(SUM(total_mb)/1024/1024,2) AS total_tb
FROM v_usage_month
GROUP BY month
ORDER BY month
""")

,month,subs,total_tb
0,2025-01,21302,337.97
1,2025-02,21162,307.32
2,2025-03,20366,332.51
3,2025-04,19320,310.27
4,2025-05,18356,363.10
5,2025-06,17326,338.95


In [102]:
q("""select round(avg(data_mb),2)as avg
  from usage_daily""")

,avg
0,1572.87


In [103]:
q("""SELECT ROUND(avg(mb_per_observed_day),2) AS avg_observed_day,
ROUND(avg(mb_per_active_day),2) AS avg_active_day
FROM customer_features;""")

,avg_observed_day,avg_active_day
0,524.05,1404.44


In [104]:
q("""
    SELECT
        strftime('%w', usage_date) AS weekday,
        COUNT(*) AS row_count,
        AVG(data_mb) AS average_data_mb
    FROM usage_daily
    GROUP BY weekday
    ORDER BY weekday
""")

 

,weekday,row_count,average_data_mb
0,0,190368,1578.056611
1,1,189914,1572.936482
2,2,183823,1569.549311
3,3,191088,1564.480661
4,4,190764,1579.999309
5,5,190542,1569.507222
6,6,190240,1575.454444


In [105]:
#D5 · How many subscribers used the network in each month?
#STEPS
#1. Read v_usage_month.
#2. Count distinct subscribers per month.
#3. Read the trend rather than any single number.
q("""
SELECT month, COUNT(DISTINCT customer_id) AS subscribers
FROM v_usage_month
GROUP BY month
ORDER BY month
""")

,month,subscribers
0,2025-01,21302
1,2025-02,21162
2,2025-03,20366
3,2025-04,19320
4,2025-05,18356
5,2025-06,17326


In [106]:
#D6 · Did voice minutes and SMS per day change from January to June? STEPS 1. Read usage_daily. 2. Bucket by month with strftime. 3. Average voice_minutes #and sms_count.
q("""
SELECT strftime('%Y-%m', usage_date) AS month,
	ROUND(AVG(voice_minutes), 2) AS avg_voice_minutes,
	ROUND(AVG(sms_count), 2) AS avg_sms_count
FROM usage_daily
WHERE usage_date >= '2025-01-01'
  AND usage_date < '2025-07-01'
GROUP BY month
ORDER BY month
""")

,month,avg_voice_minutes,avg_sms_count
0,2025-01,55.56,0.75
1,2025-02,55.84,0.75
2,2025-03,56.08,0.75
3,2025-04,56.49,0.76
4,2025-05,56.70,0.76
5,2025-06,57.13,0.77


In [107]:
#D7 · How much data did an active subscriber use each month, and on how many days?
#STEPS
#1. Read v_usage_month.
#2. Group by month.
#3. Average total_mb per subscriber and convert to gigabytes.
#4. Also average usage_days, to see whether people are appearing more often or using more when they 
#appear.
q("""
SELECT month,
    ROUND(AVG(total_mb) / 1024.0, 2) AS avg_gb_per_subscriber,
    ROUND(AVG(usage_days), 2) AS avg_usage_days
FROM v_usage_month
GROUP BY month
ORDER BY month
""")

,month,avg_gb_per_subscriber,avg_usage_days
0,2025-01,16.25,11.39
1,2025-02,14.87,10.31
2,2025-03,16.72,11.52
3,2025-04,16.44,11.25
4,2025-05,20.26,11.73
5,2025-06,20.03,11.46


In [108]:
# D8 · Among subscribers who never left, did data per active day still rise in May?
# Hold the cohort constant by keeping only subscribers who never churned.
q("""
SELECT
    v.month,
    COUNT(DISTINCT v.customer_id) AS subscribers,
    ROUND(AVG(1.0 * v.total_mb / NULLIF(v.usage_days, 0)), 2) AS avg_mb_per_active_day
FROM v_usage_month AS v
JOIN customer_features AS c
    ON c.customer_id = v.customer_id
WHERE c.is_churned = 0
GROUP BY v.month
ORDER BY v.month
""")

,month,subscribers,avg_mb_per_active_day
0,2025-01,16293,1410.58
1,2025-02,16588,1422.15
2,2025-03,16640,1418.06
3,2025-04,16648,1411.76
4,2025-05,16667,1659.45
5,2025-06,16660,1662.24


In [109]:
q("""
  
with new_total as (select count(*)as total from customer_features)
select * from new_total;

""")

,total
0,22000


In [110]:
q("""
    WITH ranked AS (
        SELECT
            NTILE(4) OVER (ORDER BY mb_per_active_day) AS quartile,
            mb_per_active_day
        FROM customer_features
        WHERE mb_per_active_day > 0
    )
    SELECT
        quartile,
        COUNT(*) AS subscribers,
        MIN(mb_per_active_day) AS min_mb_per_active_day,
        MAX(mb_per_active_day) AS max_mb_per_active_day
    FROM ranked
    GROUP BY quartile
    ORDER BY quartile
""")

,quartile,subscribers,min_mb_per_active_day,max_mb_per_active_day
0,1,5499,25.46,657.32
1,2,5499,657.35,1092.83
2,3,5499,1092.84,1571.87
3,4,5499,1572.12,9719.60


In [111]:
q("""
    SELECT
        c.connection_type,
        c.device_type,
        ROUND(AVG(u.voice_minutes), 2) AS avg_voice_minutes_per_active_day,
        ROUND(AVG(u.data_mb), 2) AS avg_mb_per_active_day,
        COUNT(*) AS active_days
    FROM usage_daily AS u
    JOIN customer_features AS c
        ON c.customer_id = u.customer_id
    GROUP BY c.connection_type, c.device_type
    ORDER BY avg_voice_minutes_per_active_day DESC
""")

,connection_type,device_type,avg_voice_minutes_per_active_day,avg_mb_per_active_day,active_days
0,Postpaid,Feature Phone,56.71,591.87,42674
1,Postpaid,Smartphone,56.33,3724.20,259745
2,Prepaid,Smartphone,56.24,1209.17,883352
3,Prepaid,Feature Phone,56.17,184.87,140968


In [112]:
q("""
select plan_name,monthly_gb,
round(avg(total_data_mb)/1024,2) as avg_gb
from customer_features
group by plan_name,monthly_gb
order by monthly_gb;""")

,plan_name,monthly_gb,avg_gb
0,Prepaid Lite 99,15.0,17.31
1,Prepaid Smart 155,30.0,34.28
2,Prepaid Value 209,30.0,35.33
3,Postpaid 399,40.0,47.13
4,Prepaid Max 479,45.0,53.06
5,Prepaid Popular 299,45.0,53.59
6,Prepaid Annual 1799,60.0,71.20
7,Prepaid Quarterly 719,60.0,70.67
8,Postpaid 499,75.0,92.82
9,Prepaid Data 349,75.0,90.76


In [113]:
q("""
  select distinct(m_score)from customer_features;""")

,m_score
0,3
1,1
2,4
3,2


In [114]:
q("""
SELECT m_score, COUNT(*) AS subs,
ROUND(AVG(recharge_total_6m),0) AS spended,
ROUND(AVG(mb_per_active_day),0) AS mb_per_day,
ROUND(100*AVG(is_churned),2) AS churn
FROM customer_features
GROUP BY m_score
ORDER BY m_score;
""")

,m_score,subs,spended,mb_per_day,churn
0,1,5500,167.0,973.0,54.82
1,2,5500,945.0,779.0,19.96
2,3,5500,1678.0,1117.0,11.24
3,4,5500,5477.0,2750.0,9.00


In [115]:
q("""
SELECT 
    ROUND(AVG(total_sms * 1.0 / NULLIF(usage_days, 0)), 2) AS sms_per_day,
    SUM(CASE WHEN total_sms = 0 THEN 1 ELSE 0 END) AS never_used,
    COUNT(*) AS subs
FROM customer_features
WHERE usage_days > 0;
""")

,sms_per_day,never_used,subs
0,0.6,654,21996


In [116]:
q("""
 select is_roaming, COUNT(*) AS subs,
 round(100*count(*)/1326739,2)as perc,
 count(distinct customer_id) as subs
 from usage_daily
group by is_roaming;
""")

,is_roaming,subs,perc,subs
0,0,1287143,97.0,21994
1,1,39596,2.0,16233


In [117]:
 # F2 · Do subscribers use more or less on roaming days than at home?
# STEPS
# 1. Same table, same grouping.
# 2. Average data_mb and voice_minutes in each group.
# 3. Read the two rows against each other.

q("""
SELECT is_roaming,
       ROUND(AVG(voice_minutes), 2) AS voice_min,
       ROUND(AVG(data_mb), 2) AS data
FROM usage_daily
GROUP BY is_roaming;

""")

,is_roaming,voice_min,data
0,0,56.27,1572.95
1,1,56.09,1570.05


In [118]:

q("""
SELECT
    connection_type,
    COUNT(*) AS subscribers,
    ROUND(AVG(roaming_days), 2) AS avg_roaming_days,
    ROUND(100.0 * AVG(CASE WHEN roaming_days > 0 THEN 1.0 ELSE 0.0 END), 2)
        AS pct_ever_roamed
FROM customer_features
GROUP BY connection_type
""")


,connection_type,subscribers,avg_roaming_days,pct_ever_roamed
0,Postpaid,4851,1.84,74.83
1,Prepaid,17149,1.79,73.49


In [119]:
q("""
SELECT
    circle_name,
    ROUND(100*SUM(roaming_days) * 1.0 / NULLIF(SUM(usage_days), 0), 2) AS roaming_per_usage_day,
    COUNT(*) AS subs
FROM customer_features
GROUP BY circle_name
ORDER BY roaming_per_usage_day DESC
LIMIT 5;
""")

,circle_name,roaming_per_usage_day,subs
0,Odisha,3.07,696
1,Haryana,3.07,1041
2,Delhi,3.07,735
3,Karnataka,3.05,1361
4,Assam,3.05,1301


In [120]:
q("""
SELECT CASE WHEN roaming_days=0 THEN '0'
WHEN roaming_days <= 2 THEN '0-2'
WHEN roaming_days <= 5 THEN '3-5'
ELSE '6+'
END AS band,
COUNT(*) AS subs,
ROUND(100.0*AVG(is_Churned),2) AS churn
FROM customer_features
WHERE usage_days>0
GROUP BY band
ORDER BY band;
""")

,band,subs,churn
0,0,5763,44.61
1,0-2,9922,21.58
2,3-5,5546,8.69
3,6+,765,3.92


In [121]:
 # G1 · What do subscribers complain about most?
# STEPS
# 1. Read complaints, where one row is one ticket.
# 2. Group by category.
# 3. Count and share.
# 4. Sort biggest first.
q("""
SELECT category, COUNT(*) AS tickets,
ROUND(100.0*COUNT(*)/(SELECT COUNT(*) FROM complaints),2) AS perc
FROM complaints
GROUP BY category
ORDER BY tickets DESC;
""")

,category,tickets,perc
0,Slow Data,8035,29.94
1,Dropped Call,6365,23.71
2,Network Coverage,4885,18.20
3,Billing,3738,13.93
4,Recharge Issue,2679,9.98
5,SIM/KYC,1139,4.24


In [122]:
# G2 · Does the complaint channel change how often a ticket is resolved?
q("""
SELECT
    channel,
    COUNT(*) AS tickets,
    ROUND(100.0 * AVG(is_resolved), 2) AS resolved_pct
FROM complaints
GROUP BY channel
ORDER BY tickets DESC
""")

,channel,tickets,resolved_pct
0,Store,6799,70.33
1,Call Center,6728,69.72
2,Twitter/X,6698,70.36
3,Airtel App,6616,70.04


In [123]:
# G3 · How many tickets are still open?
q("""
SELECT
    CASE WHEN is_resolved = 0 THEN 'Open' ELSE 'Resolved' END AS status,
    COUNT(*) AS tickets,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM complaints), 2) AS share_pct
FROM complaints
GROUP BY is_resolved
""")

,status,tickets,share_pct
0,Open,8021,29.88
1,Resolved,18820,70.12


In [124]:
q("""SELECT category,COUNT(*) AS tickets,
ROUND(100.0*SUM(is_resolved)/COUNT(*),2)AS resolved_perc,
ROUND(AVG(resolution_days),2) AS avg_resolved_days
FROM complaints
GROUP BY category
ORDER BY resolved_perc;
""")

,category,tickets,resolved_perc,avg_resolved_days
0,SIM/KYC,1139,68.57,7.65
1,Network Coverage,4885,69.52,7.60
2,Dropped Call,6365,69.71,7.46
3,Slow Data,8035,69.97,7.55
4,Billing,3738,70.87,7.50
5,Recharge Issue,2679,72.23,7.55


In [125]:
# 2 G5 · How many days does a resolved ticket take to close?
# STEPS
# 1. Filter to resolved tickets only.
# 2. Average, minimum and maximum resolution_days.
# 3. Say explicitly what population that is measured on.
q("""
SELECT
    COUNT(*) AS resolved_tickets,
    ROUND(AVG(resolution_days), 2) AS avg_resolution_days,
    MIN(resolution_days) AS min_resolution_days,
    MAX(resolution_days) AS max_resolution_days
FROM complaints
WHERE is_resolved = 1
""")

,resolved_tickets,avg_resolution_days,min_resolution_days,max_resolution_days
0,18820,7.54,1.0,14.0


In [126]:
q("""
SELECT
    CASE
        WHEN complaint_count = 0 THEN '0'
        WHEN complaint_count = 1 THEN '1'
        ELSE '2+'
    END AS complaint_band,
    COUNT(*) AS subscribers
FROM customer_features
GROUP BY complaint_band
ORDER BY CASE complaint_band
    WHEN '0' THEN 0
    WHEN '1' THEN 1
    ELSE 2
END
""")

,complaint_band,subscribers
0,0,6631
1,1,7854
2,2+,7515


In [127]:
# · Which five circles raise the most complaints per subscriber?
# STEPS
# 1. Group by circle_name.
# 2. Divide total tickets by subscribers so circles of different sizes compare.
# 3. Sort and take five
q("""
SELECT circle_name,
    COUNT(*) AS subs,
    ROUND(1.0 * SUM(complaint_count) / COUNT(*), 2) AS perc
    FROM customer_features
    GROUP BY circle_name
    ORDER BY perc DESC
    LIMIT 5;
""")

,circle_name,subs,perc
0,Tamil Nadu,731,1.29
1,Bihar,887,1.29
2,Punjab,558,1.27
3,Kolkata,572,1.27
4,Delhi,735,1.27


In [128]:
q("""

SELECT strftime('%Y',activation_date) AS year,COUNT(*) AS subs
FROM customer_features
GROUP BY year;
""")

,year,subs
0,2022,6922
1,2023,6925
2,2024,7053
3,2025,1100


In [129]:
# H2 · What share of each joining year has left?
q("""
SELECT
    strftime('%Y', activation_date) AS joined_year,
    COUNT(*) AS subscribers,
    SUM(is_churned) AS churned_subscribers,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_pct
FROM customer_features
GROUP BY joined_year
ORDER BY joined_year
""")

,joined_year,subscribers,churned_subscribers,churn_pct
0,2022,6922,1374,19.85
1,2023,6925,1632,23.57
2,2024,7053,1901,26.95
3,2025,1100,319,29.00


In [130]:
# H3 · How does churn change with each completed year on the network?
q("""
SELECT
    tenure_years,
    COUNT(*) AS subscribers,
    SUM(is_churned) AS churned_subscribers,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_pct
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years
""")

,tenure_years,subscribers,churned_subscribers,churn_pct
0,0,4610,1328,28.81
1,1,6961,1725,24.78
2,2,7055,1541,21.84
3,3,3374,632,18.73


In [131]:
 # H4 · How long had a leaver been with us when they left?
# STEPS
# 1. Filter to churned subscribers.
# 2. Average, minimum and maximum days_to_churn, which Step 2 built from the two dates.
q("""
SELECT
    COUNT(*) AS Leavers,
    AVG(days_to_churn) AS avg_days,
    MIN(days_to_churn) AS min_days,
    MAX(days_to_churn) AS max_days
FROM customer_features
WHERE is_churned = 1
""")

,Leavers,avg_days,min_days,max_days
0,5226,567.183123,10,1259


In [132]:
# Do longer-standing subscribers pay more?
q("""
SELECT
    tenure_years,
    COUNT(*) AS sub,
    ROUND(AVG(actual_arpu_6m), 2) AS avg_monthly_payment,
    ROUND(AVG(recharge_total_6m), 2) AS avg_recharge
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years
""")

,tenure_years,sub,avg_monthly_payment,avg_recharge
0,0,4610,329.10,1974.63
1,1,6961,342.72,2056.30
2,2,7055,350.13,2100.76
3,3,3374,357.22,2143.29


In [133]:
q("""
SELECT
    COUNT(*) AS total_sub,
    SUM(is_churned) AS sub_lost,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
""")

,total_sub,sub_lost,churn_rate_pct
0,22000,5226,23.75


In [134]:
# I2 · What is the churn rate for prepaid and for postpaid subscribers?
# STEPS
# 1. Group by connection_type.
# 2. Count, sum and average churn.
q("""
SELECT
    connection_type,
    COUNT(*) AS subscribers,
    SUM(is_churned) AS churned,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
GROUP BY connection_type
""")

,connection_type,subscribers,churned,churn_rate_pct
0,Postpaid,4851,785,16.18
1,Prepaid,17149,4441,25.90


In [135]:
# I3 · Which six plans have the highest churn rate?
# STEPS
# 1. Group by plan_name, carrying plan_type.
# 2. Count and average churn.
# 3. Sort worst first, take six.
q("""
SELECT plan_name, plan_type,
    COUNT(*) AS subscribers,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
GROUP BY plan_name, plan_type
ORDER BY churn_rate_pct DESC
LIMIT 6
""")

,plan_name,plan_type,subscribers,churn_rate_pct
0,Prepaid Lite 99,Prepaid,996,31.12
1,Prepaid Smart 155,Prepaid,1137,30.17
2,Prepaid Entertainment 399,Prepaid,2142,26.24
3,Prepaid Quarterly 719,Prepaid,2561,25.85
4,Prepaid Value 209,Prepaid,1437,25.26
5,Prepaid Annual 1799,Prepaid,2901,25.13


In [136]:
q("""
SELECT circle_name, region,
    COUNT(*) AS subscribers,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
GROUP BY circle_name, region
ORDER BY churn_rate_pct DESC
""")

,circle_name,region,subscribers,churn_rate_pct
0,Himachal Pradesh,North,1247,33.44
1,Maharashtra,West,1428,32.91
2,Andhra Pradesh,South,1412,28.90
3,Mumbai,West,1395,28.10
4,Uttar Pradesh (East),North,554,27.26
5,Haryana,North,1041,25.46
6,Jammu & Kashmir,North,933,24.87
7,Gujarat,West,1210,24.63
8,Assam,East,1301,23.14
9,Uttar Pradesh (West),North,1081,22.94


In [137]:
# I5 · Do smartphone users leave at a different rate from feature-phone users?
q("""
SELECT
    device_type,
    COUNT(*) AS subscribers,
    SUM(is_churned) AS churned,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
GROUP BY device_type
ORDER BY device_type
""")

,device_type,subscribers,churned,churn_rate_pct
0,Feature Phone,3029,719,23.74
1,Smartphone,18971,4507,23.76


In [138]:
# I6 · Does churn differ by acquisition channel?
q("""
SELECT
    acquisition_channel,
    COUNT(*) AS subscribers,
    SUM(is_churned) AS churned,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
GROUP BY acquisition_channel
ORDER BY churn_rate_pct DESC
""")

,acquisition_channel,subscribers,churned,churn_rate_pct
0,Call Center,1770,436,24.63
1,Airtel App,6513,1576,24.20
2,Website,3081,743,24.12
3,Doorstep,1349,321,23.80
4,Retailer,9287,2150,23.15


In [139]:
# I7 · Does churn differ by age band?
q("""
SELECT
    age_band,
    COUNT(*) AS subscribers,
    SUM(is_churned) AS churned,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
GROUP BY age_band
ORDER BY age_band
""")

,age_band,subscribers,churned,churn_rate_pct
0,18-24,5268,1246,23.65
1,25-34,7467,1770,23.70
2,35-44,4824,1144,23.71
3,45-54,2895,680,23.49
4,55+,1546,386,24.97


In [140]:
q("""
SELECT CASE WHEN complaint_count=0
            THEN 'never complained'
            WHEN unresolved_count=0
            THEN 'all resolved'
            ELSE 'open ticket'
       END AS grp,
       COUNT(*) AS subs,
       ROUND(100.0* AVG(is_churned),2) AS churn
FROM customer_features
GROUP BY grp
ORDER BY grp ;
""")

,grp,subs,churn
0,all resolved,8689,23.75
1,never complained,6631,20.03
2,open ticket,6680,27.46


In [141]:
q("""
SELECT
    unresolved_count,
    COUNT(*) AS subscribers,
    SUM(is_churned) AS churned,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
GROUP BY unresolved_count
ORDER BY unresolved_count
""")

,unresolved_count,subscribers,churned,churn_rate_pct
0,0,15320,3392,22.14
1,1,5515,1438,26.07
2,2,1018,337,33.10
3,3,120,45,37.50
4,4,25,12,48.00
5,5,2,2,100.00


In [142]:
# K1 · Churn rate in each recency quartile
q("""
SELECT
    r_score,
    COUNT(*) AS subs,
    SUM(is_churned) AS churned,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
GROUP BY r_score
ORDER BY r_score
""")

,r_score,subs,churned,churn_rate_pct
0,1,5500,5032,91.49
1,2,5500,194,3.53
2,3,5500,0,0.00
3,4,5500,0,0.00


In [143]:
# K2 · What is the churn rate in each frequency quartile?
# Check the average recharge_count and churn rate for each f_score.
q("""
SELECT
    f_score,
    COUNT(*) AS subscribers,
    ROUND(AVG(recharge_count), 2) AS avg_recharge_count,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
GROUP BY f_score
ORDER BY f_score
""")

,f_score,subscribers,avg_recharge_count,churn_rate_pct
0,1,5500,0.33,45.44
1,2,5500,2.51,38.67
2,3,5500,5.46,7.76
3,4,5500,6.92,3.15


In [144]:
# K3 · Spend and churn rate in each spend quartile
q("""
SELECT
    m_score,
    COUNT(*) AS subscribers,
    ROUND(AVG(recharge_total_6m), 2) AS avg_spend,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
GROUP BY m_score
ORDER BY m_score
""")

,m_score,subscribers,avg_spend,churn_rate_pct
0,1,5500,166.77,54.82
1,2,5500,945.26,19.96
2,3,5500,1678.08,11.24
3,4,5500,5477.03,9.00


In [145]:
# L1 · Churn rate and monthly revenue by region
q("""
SELECT
    region,
    COUNT(*) AS subs,
    ROUND(SUM(actual_arpu_6m), 2) AS mon_revenue,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
GROUP BY region
ORDER BY churn_rate_pct 
""")

,region,subs,mon_revenue,churn_rate_pct
0,South,4645,1635737.25,21.85
1,East,5459,1893199.66,21.91
2,North,7231,2477493.21,23.73
3,West,4665,1571780.02,27.85


In [146]:
# L2 · Which four circles churn the most, and what do their subscribers pay?
q("""
SELECT
    circle_name,
    region,
    COUNT(*) AS subscribers,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct,
    ROUND(AVG(actual_arpu_6m), 2) AS avg_monthly_payment
FROM customer_features
GROUP BY circle_name, region
ORDER BY churn_rate_pct DESC
LIMIT 4
""")

,circle_name,region,subscribers,churn_rate_pct,avg_monthly_payment
0,Himachal Pradesh,North,1247,33.44,313.16
1,Maharashtra,West,1428,32.91,315.15
2,Andhra Pradesh,South,1412,28.90,328.17
3,Mumbai,West,1395,28.10,339.27


In [147]:
# L7 · Do regions differ in revenue per subscriber and in data use?
q("""
SELECT
    region,
    COUNT(*) AS subscribers,
    ROUND(AVG(actual_arpu_6m), 2) AS avg_monthly_revenue_per_subscriber,
    ROUND(AVG(mb_per_active_day), 2) AS avg_mb_per_active_day
FROM customer_features
GROUP BY region
ORDER BY avg_monthly_revenue_per_subscriber DESC
""")

,region,subscribers,avg_monthly_revenue_per_subscriber,avg_mb_per_active_day
0,South,4645,352.15,1430.50
1,East,5459,346.80,1404.47
2,North,7231,342.62,1388.55
3,West,4665,336.93,1403.09


In [148]:
q("""
SELECT
    ab_group,
    COUNT(*) AS subscribers,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customer_features), 2) AS share_pct
FROM customer_features
GROUP BY ab_group
ORDER BY ab_group
""")

,ab_group,subscribers,share_pct
0,A,10996,49.98
1,B,11004,50.02


In [149]:
q("""
SELECT
    ab_group,
    COUNT(*) AS subscribers,
    SUM(is_churned) AS churned,
    ROUND(100.0 * AVG(is_churned), 2) AS churn_rate_pct
FROM customer_features
GROUP BY ab_group
ORDER BY ab_group
""")

,ab_group,subscribers,churned,churn_rate_pct
0,A,10996,2587,23.53
1,B,11004,2639,23.98


In [150]:
q("""
SELECT
    ab_group,
    COUNT(*) AS subscribers,
    ROUND(AVG(stated_arpu), 2) AS avg_stated_arpu,
    ROUND(AVG(actual_arpu_6m), 2) AS avg_actual_arpu
FROM customer_features
GROUP BY ab_group
ORDER BY ab_group
""")

,ab_group,subscribers,avg_stated_arpu,avg_actual_arpu
0,A,10996,424.30,347.58
1,B,11004,418.99,341.35
